In [1]:
import sys
from types import MethodType

sys.path.append(r"C:\Users\AlexanderZirn\Documents\GIT\nRTD\lib")
from pathlib import Path
import argparse

import numpy as np
import torch
import torch.utils
import torch.utils.data
import wandb
import lightning.pytorch as pl
from lightning.pytorch.loggers.wandb import WandbLogger
from lightning.pytorch.callbacks import LearningRateMonitor

torch.set_default_dtype(torch.float64)

from nRTD import RTDModule, RTDDataModule
#from SweepRunner import Sweeper
import matplotlib.pyplot as plt

plt.style.use("ICIWstyle")

In [2]:
def log_plots(data_module: RTDDataModule, model: RTDModule, index=0, Es_to_plot=[0]):
    pred_y, pred_t = model(data_module.x, data_module.t_in)
    pred_t = pred_t.detach().numpy().squeeze()
    pred_y = pred_y.detach().numpy().squeeze()

    fig = plt.figure()
    index = 0
    plt.plot(
        data_module.t_in[index],
        data_module.x[index].squeeze(),
        ls="--",
    )
    plt.plot(
        data_module.t_in[index + 2],
        data_module.x[index + 2].squeeze(),
        ls="--",
    )
    plt.plot(
        data_module.t_out[index],
        data_module.y[index].squeeze(),
        "o",
        c="C0",
        markersize=0.5,
    )
    plt.plot(
        data_module.t_out[index + 2],
        data_module.y[index + 2].squeeze(),
        "o",
        c="C1",
        markersize=0.5,
    )
    plt.plot(
        data_module.t_in[index + 1],
        data_module.x[index + 1].squeeze(),
        ls="--",
        c="C2",
    )
    plt.plot(
        data_module.t_in[index + 3],
        data_module.x[index + 3].squeeze(),
        ls="--",
        c="C3",
    )
    plt.plot(
        data_module.t_out[index + 1],
        data_module.y[index + 1].squeeze(),
        "o",
        c="C2",
        markersize=0.5,
    )
    plt.plot(
        data_module.t_out[index + 3],
        data_module.y[index + 3].squeeze(),
        "o",
        c="C3",
        markersize=0.5,
    )

    plt.plot(pred_t[index, :], pred_y[index, :], c="C0")
    plt.plot(pred_t[index + 2].squeeze(), pred_y[index + 2].squeeze(), c="C1")
    plt.plot(pred_t[index + 1, :], pred_y[index + 1, :], c="C2")
    plt.plot(pred_t[index + 3].squeeze(), pred_y[index + 3].squeeze(), c="C3")
    plt.xlabel("t / s")
    plt.ylabel("x / 1")
    plt.twinx()
    ax = plt.gca()
    plt.ylabel("$E \;/\; s^{-1}$")
    my_colors = ["darkmagenta", "goldenrod", "firebrick", "darkcyan"]
    for E_index in Es_to_plot:
        ax.plot(
            model.net.conv_layers[E_index].t_kernel.detach().numpy(),
            model.net.E[E_index],
            color=my_colors[E_index],
        )
    wandb.log({f"RTD_Plot_img": wandb.Image(fig)})
    wandb.log({f"RTD_Plot_fig": fig})

In [3]:
### Constants
# givens
t_out = (0.0, 120.0)
print(f"t_out: {t_out}")
n_out = int(4 * (t_out[1] - t_out[0]) + 1)
print(f"n_out: {n_out}")
delta_t_out = (t_out[1] - t_out[0]) / (n_out - 1)
print(f"delta_t_out: {delta_t_out}")
# # the capillary times
# # FBA_18032025_300ml-min_Analytik_MS
# switching_periods = np.array(
#     [
#         5 * 60 + 0.03,
#         5 * 60 + 0.27,
#         5 * 60 + 0.27,
#         5 * 60 + 0.28,
#         5 * 60 + 0.18,
#         5 * 60 + 0.26,
#         5 * 60 + 0.33,
#         5 * 60 + 0.27,
#         5 * 60 + 0.18,
#         5 * 60 + 0.05,
#     ]
# )
# switching_times_cap = np.flip(np.cumsum(-switching_periods))
# switching_times_cap = np.append(switching_times_cap, 0) - 1

# the piping times
t_halfperiod = t_out[1] - t_out[0]
print(f"t_halfperiod: {t_halfperiod}")

# automated_switching_times = np.full((20,), t_halfperiod)
# switching_times_reac_inlet = np.insert(automated_switching_times, 0, 7.4 - 1.0)
# switching_times_reac_inlet = np.cumsum(switching_times_reac_inlet)
# switching_times_reac_outlet = np.insert(automated_switching_times, 0, 7.5 - 1.0)
# switching_times_reac_outlet = np.cumsum(switching_times_reac_outlet)
# switching_times_total_system = np.insert(automated_switching_times, 0, 7.7 - 1.0)
# switching_times_total_system = np.cumsum(switching_times_total_system)

t_kernels = [
    (0.0, 15.0)#,#30  # piping to reactor inlet
    #(0.0, 40.0),  # reactor
    #(0.0, 60.0)#,  # piping after reactor outlet
    #(0.0, 30.0),  # capillary
]
print(f"t_kernels: {t_kernels}")

n_kernels = list(
    map(
        lambda t_kernel: int(((t_kernel[1] - t_kernel[0]) / delta_t_out) + 1), t_kernels
    )
)
print(f"n_kernels: {n_kernels}")
t_in = (0, t_out[1] - sum([t_kernel[1] for t_kernel in t_kernels]))
print(f"t_in: {t_in}")
n_in = int(((t_in[1] - t_in[0]) / delta_t_out) + 1)
print(f"n_in: {n_in}")

t_out: (0.0, 120.0)
n_out: 481
delta_t_out: 0.25
t_halfperiod: 120.0
t_kernels: [(0.0, 15.0)]
n_kernels: [61]
t_in: (0, 105.0)
n_in: 421


In [4]:
# create array with half-period times

t_offset = 3600 + 5 * 60 + 30 # offset until experiment starts
t_halfperiod = 300 # 5min half-period duration

switching_times = np.full((24,), t_halfperiod) #+ np.array([0.05,
                                                        #   0.18,
                                                        #   0.27,
                                                        #   0.33,
                                                        #   0.26,
                                                        #   0.18,
                                                        #   0.28,
                                                        #   0.27,
                                                        #   0.27,
                                                        #   0.03])# 24 half-periods
print(switching_times)
switching_times = np.insert(switching_times, 0, t_offset)
switching_times = np.cumsum(switching_times)
print(switching_times)

intervals = [
    (switching_times[i], switching_times[i + 1])
    for i in range(len(switching_times) - 1)
]
print(intervals)

[300 300 300 300 300 300 300 300 300 300 300 300 300 300 300 300 300 300
 300 300 300 300 300 300]
[ 3930  4230  4530  4830  5130  5430  5730  6030  6330  6630  6930  7230
  7530  7830  8130  8430  8730  9030  9330  9630  9930 10230 10530 10830
 11130]
[(np.int64(3930), np.int64(4230)), (np.int64(4230), np.int64(4530)), (np.int64(4530), np.int64(4830)), (np.int64(4830), np.int64(5130)), (np.int64(5130), np.int64(5430)), (np.int64(5430), np.int64(5730)), (np.int64(5730), np.int64(6030)), (np.int64(6030), np.int64(6330)), (np.int64(6330), np.int64(6630)), (np.int64(6630), np.int64(6930)), (np.int64(6930), np.int64(7230)), (np.int64(7230), np.int64(7530)), (np.int64(7530), np.int64(7830)), (np.int64(7830), np.int64(8130)), (np.int64(8130), np.int64(8430)), (np.int64(8430), np.int64(8730)), (np.int64(8730), np.int64(9030)), (np.int64(9030), np.int64(9330)), (np.int64(9330), np.int64(9630)), (np.int64(9630), np.int64(9930)), (np.int64(9930), np.int64(10230)), (np.int64(10230), np.int64(1053

In [5]:
wandb.init(project="HSA_MGA_nRTD", entity="ice_ulm")

wandb: Currently logged in as: alexander-zirn (ice_ulm) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [6]:
import sys

sys.path.append(r"C:\Users\AlexanderZirn\Documents\GIT\nRTD\lib")
from pathlib import Path
from nRTD.rtd_fitting import RTDDataModule

data_reac_inlet = RTDDataModule(
    batch_size= 32, # 2 temperatures, 2 half-periods, 4 different capillary positions
    data_file=Path(
        r"C:\Users\AlexanderZirn\Documents\GIT\nRTD\Experiments\AZA_001\B - upstream\AZA-E-130625_250ml-min_2bar_250_upstream.npz"
    ),
    switching_times=switching_times,
    t_range_in=t_in,
    n_in=n_in,
    t_range_out=t_out,
    n_out=n_out,
    switch_delay=0.001,
)

In [7]:
model = RTDModule(
    kernel_sizes = n_kernels,
    kernel_times = t_kernels,
    learning_rate = 4e-3,
    use_scheduler = False,
    scheduler_kwargs = {"factor": 0.6, "patience": 2000},
)

In [8]:
# #%pip install litmodels
#from litmodels import LitModelCheckpoint
# #import litmodels

my_logger = WandbLogger(log_model=True)
lr_callback = LearningRateMonitor()

max_epochs = 20_000

trainer = pl.Trainer(
    accelerator="cpu",
    max_epochs=max_epochs,
    enable_progress_bar=True,
    logger=my_logger,
    callbacks=[lr_callback],
)

Using default `ModelCheckpoint`. Consider installing `litmodels` package to enable `LitModelCheckpoint` for automatic upload to the Lightning model registry.
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


In [9]:
model.configure_optimizers()

{'optimizer': Adam (
 Parameter Group 0
     amsgrad: False
     betas: (0.9, 0.999)
     capturable: False
     decoupled_weight_decay: False
     differentiable: False
     eps: 1e-08
     foreach: None
     fused: None
     lr: 0.004
     maximize: False
     weight_decay: 0
 )}

In [10]:
trainer.fit(model, data_reac_inlet)

C:\Users\AlexanderZirn\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\lightning\pytorch\trainer\configuration_validator.py:70: You defined a `validation_step` but have no `val_dataloader`. Skipping val loop.
C:\Users\AlexanderZirn\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\lightning\pytorch\loggers\wandb.py:397: There is a wandb run already in progress and newly created instances of `WandbLogger` will reuse this run. If this is not desired, call `wandb.finish()` before instantiating `WandbLogger`.

  | Name | Type   | Params | Mode 
----------------------------------------
0 | net  | RTDNet | 61     | train
----------------------------------------
61        Trainable params
0         Non-trainable params
61        Total params
0.000     Total estimated model params size (MB)
4         Modules in train mode
0         Modules in eva

C:\Users\AlexanderZirn\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:425: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=21` in the `DataLoader` to improve performance.
C:\Users\AlexanderZirn\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\lightning\pytorch\loops\fit_loop.py:310: The number of training batches (2) is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower value for log_every_n_steps if you want to see logs for the training epoch.


Epoch 19999: 100%|██████████| 2/2 [00:00<00:00, 59.15it/s, v_num=ljla, train/loss=3.9e-6]   

`Trainer.fit` stopped: `max_epochs=20000` reached.


Epoch 19999: 100%|██████████| 2/2 [00:00<00:00, 41.72it/s, v_num=ljla, train/loss=3.9e-6]


In [11]:
%pip install plotly


[notice] A new release of pip is available: 24.0 -> 25.1.1


[notice] To update, run: C:\Users\AlexanderZirn\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [12]:
log_plots(
    data_module=data_reac_inlet,
    model=model,
    index=0,
    Es_to_plot=[0]#, 1, 2, 3],
)
plt.savefig("RTD upstream.svg")
plt.show()

<Figure size 640x480 with 0 Axes>